In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

import torch
from torch.utils.data import Dataset, DataLoader,WeightedRandomSampler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler


: 

## Generate Dataset

In [ ]:
img_size=10
img=np.zeros((img_size,img_size))

# All Dataset

In [64]:


data=pd.read_csv('/home/tatyana/Work/LearnStuff/Embedding_for_Yuriy/Yuriy_dataset/Multicellular_dataset.csv')
freq=data['primary_class'].value_counts(normalize=True)
mapping = {cls: i for i, cls in enumerate(freq.index)}

#==============================================#
### use subset of data for calculation
#==============================================#

#select portion of the data
df, _ = train_test_split(data, 
                        train_size=0.20,  
                        stratify=data['primary_class'],shuffle=True,random_state=42)    
print(df.shape)



(269608, 37)


In [65]:
print(type(df))
# Output: <class 'pandas.core.frame.DataFrame'>

<class 'pandas.core.frame.DataFrame'>


In [84]:
class TabularDataset_conv_feature(Dataset):
        def __init__(self, X_signal,X_feature, Y):
            self.X_signal = torch.tensor(X_signal, dtype=torch.float32)
            self.X_feature = torch.tensor(X_feature, dtype=torch.float32)
            self.Y = torch.tensor(Y, dtype=torch.long)
        
        def __len__(self):
            return len(self.X_feature)
        
        def __getitem__(self, idx):
            return self.X_signal[idx].unsqueeze(0),self.X_feature[idx], self.Y[idx]

In [92]:
#==============================================#
## X preparation
#==============================================#
columns_list = ['b_0', 'b_1', 'b_2', 'b_3', 'b_4', 'b_5', 'b_6', 'b_7', 'b_8', 'b_9', 
                'b_10', 'b_11', 'b_12', 'b_13', 'b_14', 'b_15', 'b_16']  #'frag_factor']

X_feature=df['frag_factor'].copy()
X_feature=X_feature.values
X_signal = df[columns_list].copy()

# normilize each signal to signal maximum
max_value = X_signal.max(axis=1)
X_signal.iloc[:, 0:17] = X_signal.div(max_value, axis=0)
X_signal=X_signal.values

# combine 2 feature fraq_factor and max_peak value
X_feature=np.stack((X_feature,max_value),axis=1)
X_feature.shape

(269608, 2)

In [93]:

#==============================================#
## Y preparation
#==============================================#
Y = df['primary_class'].map(mapping).values

#print('class mapping')
# for k, v in mapping.items():
#     name=k.replace("class_","")
#     print(f"{v} {freq.loc[k].item():.3f} {name} ")


#==============================================#
### Torch Dataset and DataLoader
#==============================================#
indices = np.arange(len(Y))

# Split just the indices (and Y, so you can still stratify)
train_idx, val_idx, Y_train, Y_val = train_test_split(indices, Y, test_size=0.2, stratify=Y, random_state=42)

X_signal_train, X_signal_val = X_signal[train_idx], X_signal[val_idx]
X_feature_train, X_feature_val = X_feature[train_idx], X_feature[val_idx]


In [112]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_feature_train_tr=scaler.fit_transform(X_feature_train)
X_feature_val_test=scaler.transform(X_feature_val)

print("Mean:", scaler.mean_)
print("Standard deviation:", scaler.scale_)
print("Variance:", scaler.var_)
print("Number of features:", scaler.n_features_in_)
print("Number of samples seen:", scaler.n_samples_seen_)

Mean: [ 0.53988311 11.50740049]
Standard deviation: [ 0.23887582 18.90920411]
Variance: [5.70616577e-02 3.57558000e+02]
Number of features: 2
Number of samples seen: 215686.0


In [114]:
X_feature_train_tr.mean(axis=0)

array([-1.38879951e-14, -2.53523755e-15])

In [113]:
X_feature_val_test.mean(axis=0)

array([-0.00402496,  0.00476477])

In [109]:
X_feature_train.std(axis=0)

array([ 0.23887582, 18.90920411])

In [94]:
# Apply those split indices to your multimodal arrays
batch_size=312
train_dataset = TabularDataset_conv_feature(X_signal_train,X_feature_train, Y_train)
val_dataset = TabularDataset_conv_feature(X_signal_val,X_feature_val, Y_val)

train_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=batch_size,shuffle=False)

print(f'train_data={len(train_dataset)},val_data={len(val_dataset)},all_data={data.shape[0] }')
print(f'nbatches_train={len(train_loader)},nbatches_val={len(val_loader)}')

train_data=215686,val_data=53922,all_data=1348041
nbatches_train=692,nbatches_val=173
